# 工業瑕疵檢測與異常定位｜完整評估實作
同時比較有對齊模板的 OpenCV 與單張輸入的 YOLO11n，涵蓋六類瑕疵、光照擾動、材質分層、框與檢驗品兩種漏檢，以及良率與誤報報告。

**三種用法**　① 第 0 節直接重算你上傳的歷史結果，只需 CPU。② 在 Colab 上傳既有 pcb_model_results.zip，執行全部即可用既有 best.pt 重新選驗證門檻並跑完整測試。③ 改 TRAIN_NEW=True 可從公開預訓練權重重新訓練。第 1 節以後需網路取得 DeepPCB 與套件，建議 GPU。每次從設定格開始會建立全新輸出目錄。

## 0｜先查看已取得的實測數字
以下 CSV 為前次成果包的逐圖框統計與逐類 AP 快照，可直接重算。整張圖只要有預測框就判 NG；部分瑕疵漏檢與整張圖被錯誤放行是兩個不同事件。此處一張圖視為一個檢驗單位；若一塊板有多張裁切影像，計算整板良率前需按實體板號彙整。

In [ ]:
import io
import pandas as pd
import numpy as np
def match_boxes(pred, gt, cutoff):
    if not len(pred) or not len(gt): return [],len(pred),len(gt)
    a=pred[:,None,:]; b=gt[None,:,:4]
    wh=np.maximum(0,np.minimum(a[:,:,2:],b[:,:,2:])-np.maximum(a[:,:,:2],b[:,:,:2])+1)
    inter=wh.prod(axis=2)
    area_a=((pred[:,2:]-pred[:,:2]+1).prod(axis=1))[:,None]
    area_b=((gt[:,2:4]-gt[:,:2]+1).prod(axis=1))[None,:]
    iou=inter/(area_a+area_b-inter)
    valid=iou>cutoff
    # 優先最大化有效配對數；IoU 僅作次要排序。
    weight=valid*(min(len(pred),len(gt))+1+iou)
    pi,gi=linear_sum_assignment(-weight)
    matches=[(int(i),int(j)) for i,j in zip(pi,gi) if valid[i,j]]
    return matches,len(pred)-len(matches),len(gt)-len(matches)

def scores(tp,fp,fn):
    precision=tp/(tp+fp) if tp+fp else 0.0
    recall=tp/(tp+fn) if tp+fn else 0.0
    return dict(TP=tp,FP=fp,FN=fn,precision=precision,recall=recall,
                f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)



def ratio(n,d):
    return n/d if d else np.nan

def summarize_group(g):
    tp,fp,fn = (int(g[k].sum()) for k in ('TP','FP','FN'))
    bad = g.gt_count > 0
    rejected = g.pred_count > 0
    good, passed = ~bad, ~rejected
    return dict(images=len(g),gt_good=int(good.sum()),gt_bad=int(bad.sum()),
                predicted_pass=int(passed.sum()),gt_yield=good.mean(),pass_rate=passed.mean(),
                escapes=int((bad & passed).sum()),escape_rate=ratio((bad & passed).sum(),bad.sum()),
                false_rejects=int((good & rejected).sum()),false_reject_rate=ratio((good & rejected).sum(),good.sum()),
                images_with_missed_boxes=int((g.FN>0).sum()),images_with_extra_boxes=int((g.FP>0).sum()),
                **scores(tp,fp,fn),fp_per_image=fp/len(g),missed_box_rate=ratio(fn,tp+fn))

def grouped(frame,keys,func):
    output=[]
    for values,g in frame.groupby(keys,sort=False,dropna=False):
        values=values if isinstance(values,tuple) else (values,)
        output.append({**dict(zip(keys,values)),**func(g)})
    return pd.DataFrame(output)

history=pd.read_csv(io.StringIO('id,TP,FP,FN,precision,recall,f1\n20085291,9,0,0,1.0,1.0,1.0\n20085292,7,0,0,1.0,1.0,1.0\n20085293,7,0,0,1.0,1.0,1.0\n20085294,9,0,0,1.0,1.0,1.0\n20085295,7,1,0,0.875,1.0,0.9333333333333333\n20085296,8,0,1,1.0,0.8888888888888888,0.9411764705882353\n20085297,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n20085298,6,0,0,1.0,1.0,1.0\n20085299,7,0,1,1.0,0.875,0.9333333333333333\n20085300,8,0,0,1.0,1.0,1.0\n20085301,8,1,0,0.8888888888888888,1.0,0.9411764705882353\n20085302,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n20085303,6,1,1,0.8571428571428571,0.8571428571428571,0.8571428571428571\n20085304,7,0,1,1.0,0.875,0.9333333333333333\n20085305,8,0,0,1.0,1.0,1.0\n20085306,6,0,0,1.0,1.0,1.0\n20085307,4,0,1,1.0,0.8,0.8888888888888888\n20085308,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n20085309,9,1,0,0.9,1.0,0.9473684210526315\n20085310,10,0,0,1.0,1.0,1.0\n20085311,1,1,1,0.5,0.5,0.5\n20085312,4,0,0,1.0,1.0,1.0\n20085313,6,0,0,1.0,1.0,1.0\n20085314,7,0,0,1.0,1.0,1.0\n20085315,7,0,0,1.0,1.0,1.0\n20085316,5,0,0,1.0,1.0,1.0\n20085317,9,0,0,1.0,1.0,1.0\n20085318,7,1,0,0.875,1.0,0.9333333333333333\n20085319,3,0,1,1.0,0.75,0.8571428571428571\n20085320,2,0,0,1.0,1.0,1.0\n20085321,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n20085322,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n20085323,7,0,0,1.0,1.0,1.0\n20085324,5,0,0,1.0,1.0,1.0\n13000198,4,0,0,1.0,1.0,1.0\n13000199,8,0,0,1.0,1.0,1.0\n13000200,6,0,0,1.0,1.0,1.0\n13000201,5,0,0,1.0,1.0,1.0\n13000202,6,0,0,1.0,1.0,1.0\n13000203,7,0,0,1.0,1.0,1.0\n13000204,7,0,1,1.0,0.875,0.9333333333333333\n13000205,7,0,1,1.0,0.875,0.9333333333333333\n13000206,4,0,0,1.0,1.0,1.0\n13000207,4,0,0,1.0,1.0,1.0\n13000208,7,0,0,1.0,1.0,1.0\n13000209,6,0,0,1.0,1.0,1.0\n13000210,6,0,0,1.0,1.0,1.0\n13000211,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n13000212,5,0,0,1.0,1.0,1.0\n13000213,3,0,0,1.0,1.0,1.0\n13000214,5,0,0,1.0,1.0,1.0\n13000215,7,0,2,1.0,0.7777777777777778,0.875\n00041200,5,0,0,1.0,1.0,1.0\n00041201,5,0,0,1.0,1.0,1.0\n00041202,4,0,0,1.0,1.0,1.0\n00041203,5,0,0,1.0,1.0,1.0\n00041204,6,0,0,1.0,1.0,1.0\n00041205,5,0,0,1.0,1.0,1.0\n00041206,4,1,1,0.8,0.8,0.8\n00041207,4,0,0,1.0,1.0,1.0\n00041208,4,0,0,1.0,1.0,1.0\n00041209,4,0,1,1.0,0.8,0.8888888888888888\n00041210,5,0,0,1.0,1.0,1.0\n00041211,3,0,1,1.0,0.75,0.8571428571428571\n00041212,5,0,0,1.0,1.0,1.0\n00041213,4,0,1,1.0,0.8,0.8888888888888888\n00041214,5,0,0,1.0,1.0,1.0\n00041215,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n00041216,6,0,0,1.0,1.0,1.0\n00041217,6,0,0,1.0,1.0,1.0\n00041218,6,0,0,1.0,1.0,1.0\n00041219,5,0,0,1.0,1.0,1.0\n00041220,6,0,0,1.0,1.0,1.0\n77000100,4,0,0,1.0,1.0,1.0\n77000101,4,0,0,1.0,1.0,1.0\n77000102,5,0,0,1.0,1.0,1.0\n77000103,5,0,0,1.0,1.0,1.0\n77000104,4,1,0,0.8,1.0,0.8888888888888888\n77000105,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n77000106,5,0,0,1.0,1.0,1.0\n92000111,3,0,0,1.0,1.0,1.0\n92000112,7,0,0,1.0,1.0,1.0\n92000113,4,0,0,1.0,1.0,1.0\n92000114,5,0,0,1.0,1.0,1.0\n92000115,6,3,1,0.6666666666666666,0.8571428571428571,0.75\n92000116,6,0,0,1.0,1.0,1.0\n92000117,7,0,1,1.0,0.875,0.9333333333333333\n92000118,4,0,0,1.0,1.0,1.0\n92000119,6,0,0,1.0,1.0,1.0\n50600050,5,0,0,1.0,1.0,1.0\n50600051,5,0,0,1.0,1.0,1.0\n50600052,5,0,0,1.0,1.0,1.0\n50600053,5,0,0,1.0,1.0,1.0\n50600054,6,0,0,1.0,1.0,1.0\n50600055,4,0,0,1.0,1.0,1.0\n50600056,5,0,0,1.0,1.0,1.0\n50600057,6,0,0,1.0,1.0,1.0\n50600058,5,0,0,1.0,1.0,1.0\n50600059,5,0,0,1.0,1.0,1.0\n50600060,6,0,0,1.0,1.0,1.0\n50600061,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n50600062,6,0,0,1.0,1.0,1.0\n50600063,4,0,0,1.0,1.0,1.0\n50600064,4,0,0,1.0,1.0,1.0\n50600065,4,0,1,1.0,0.8,0.8888888888888888\n50600066,4,1,1,0.8,0.8,0.8\n50600067,6,0,0,1.0,1.0,1.0\n50600068,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n50600069,5,0,0,1.0,1.0,1.0\n50600070,4,0,0,1.0,1.0,1.0\n50600071,6,0,0,1.0,1.0,1.0\n50600072,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n50600073,5,0,0,1.0,1.0,1.0\n50600074,5,0,0,1.0,1.0,1.0\n50600075,5,0,0,1.0,1.0,1.0\n50600076,6,0,0,1.0,1.0,1.0\n50600077,5,0,0,1.0,1.0,1.0\n50600078,6,0,0,1.0,1.0,1.0\n44000011,8,0,0,1.0,1.0,1.0\n44000012,9,0,1,1.0,0.9,0.9473684210526315\n44000013,10,0,0,1.0,1.0,1.0\n44000014,6,0,0,1.0,1.0,1.0\n44000015,9,0,0,1.0,1.0,1.0\n44000016,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n44000017,3,0,3,1.0,0.5,0.6666666666666666\n44000018,7,0,0,1.0,1.0,1.0\n44000019,7,0,1,1.0,0.875,0.9333333333333333\n44000020,1,3,8,0.25,0.1111111111111111,0.15384615384615385\n44000021,7,0,0,1.0,1.0,1.0\n44000022,6,0,0,1.0,1.0,1.0\n44000023,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n44000024,10,0,0,1.0,1.0,1.0\n44000025,8,2,0,0.8,1.0,0.8888888888888888\n44000026,7,1,0,0.875,1.0,0.9333333333333333\n44000027,7,0,0,1.0,1.0,1.0\n44000028,8,1,1,0.8888888888888888,0.8888888888888888,0.8888888888888888\n44000029,7,1,1,0.875,0.875,0.875\n44000030,6,0,0,1.0,1.0,1.0\n44000031,5,1,4,0.8333333333333334,0.5555555555555556,0.6666666666666666\n44000032,5,1,2,0.8333333333333334,0.7142857142857143,0.7692307692307693\n44000033,7,1,0,0.875,1.0,0.9333333333333333\n44000034,8,0,2,1.0,0.8,0.8888888888888888\n44000036,4,0,2,1.0,0.6666666666666666,0.8\n44000037,4,0,2,1.0,0.6666666666666666,0.8\n44000038,7,0,0,1.0,1.0,1.0\n44000039,7,0,0,1.0,1.0,1.0\n44000040,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n44000041,7,0,0,1.0,1.0,1.0\n44000042,4,0,2,1.0,0.6666666666666666,0.8\n44000043,5,1,1,0.8333333333333334,0.8333333333333334,0.8333333333333334\n44000044,5,1,1,0.8333333333333334,0.8333333333333334,0.8333333333333334\n44000045,4,0,3,1.0,0.5714285714285714,0.7272727272727273\n44000046,5,1,2,0.8333333333333334,0.7142857142857143,0.7692307692307693\n44000047,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n44000048,6,0,0,1.0,1.0,1.0\n44000049,6,0,0,1.0,1.0,1.0\n44000050,7,1,0,0.875,1.0,0.9333333333333333\n44000051,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n44000052,6,0,3,1.0,0.6666666666666666,0.8\n44000053,4,0,2,1.0,0.6666666666666666,0.8\n44000054,7,0,0,1.0,1.0,1.0\n44000055,7,0,2,1.0,0.7777777777777778,0.875\n44000056,7,1,2,0.875,0.7777777777777778,0.8235294117647058\n44000057,8,0,0,1.0,1.0,1.0\n44000058,7,1,0,0.875,1.0,0.9333333333333333\n44000059,7,1,1,0.875,0.875,0.875\n44000060,7,0,1,1.0,0.875,0.9333333333333333\n44000061,8,1,1,0.8888888888888888,0.8888888888888888,0.8888888888888888\n90100000,6,0,0,1.0,1.0,1.0\n90100001,7,0,0,1.0,1.0,1.0\n90100002,6,0,0,1.0,1.0,1.0\n90100003,6,1,1,0.8571428571428571,0.8571428571428571,0.8571428571428571\n90100004,8,1,0,0.8888888888888888,1.0,0.9411764705882353\n90100005,7,1,1,0.875,0.875,0.875\n90100006,4,2,2,0.6666666666666666,0.6666666666666666,0.6666666666666666\n90100008,8,0,2,1.0,0.8,0.8888888888888888\n90100009,10,2,3,0.8333333333333334,0.7692307692307693,0.8\n90100010,11,2,1,0.8461538461538461,0.9166666666666666,0.88\n90100011,9,0,1,1.0,0.9,0.9473684210526315\n90100012,8,4,1,0.6666666666666666,0.8888888888888888,0.7619047619047619\n90100013,7,2,2,0.7777777777777778,0.7777777777777778,0.7777777777777778\n90100014,8,0,1,1.0,0.8888888888888888,0.9411764705882353\n90100016,8,0,1,1.0,0.8888888888888888,0.9411764705882353\n90100017,7,1,1,0.875,0.875,0.875\n90100018,8,0,0,1.0,1.0,1.0\n90100019,8,0,2,1.0,0.8,0.8888888888888888\n90100020,6,0,2,1.0,0.75,0.8571428571428571\n90100021,7,0,0,1.0,1.0,1.0\n90100022,5,2,1,0.7142857142857143,0.8333333333333334,0.7692307692307693\n90100024,6,0,0,1.0,1.0,1.0\n90100025,6,0,0,1.0,1.0,1.0\n90100026,4,2,1,0.6666666666666666,0.8,0.7272727272727273\n90100027,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n90100028,3,0,1,1.0,0.75,0.8571428571428571\n90100029,4,0,1,1.0,0.8,0.8888888888888888\n90100030,3,2,2,0.6,0.6,0.6\n90100032,5,0,0,1.0,1.0,1.0\n90100033,3,0,3,1.0,0.5,0.6666666666666666\n90100035,6,1,1,0.8571428571428571,0.8571428571428571,0.8571428571428571\n90100036,4,2,1,0.6666666666666666,0.8,0.7272727272727273\n90100037,5,1,1,0.8333333333333334,0.8333333333333334,0.8333333333333334\n90100038,5,0,0,1.0,1.0,1.0\n90100040,5,0,0,1.0,1.0,1.0\n90100041,4,0,1,1.0,0.8,0.8888888888888888\n90100042,3,6,2,0.3333333333333333,0.6,0.42857142857142855\n90100043,4,0,2,1.0,0.6666666666666666,0.8\n90100044,3,1,2,0.75,0.6,0.6666666666666666\n90100045,4,0,2,1.0,0.6666666666666666,0.8\n90100046,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n90100048,6,0,0,1.0,1.0,1.0\n90100049,6,0,0,1.0,1.0,1.0\n90100050,4,0,1,1.0,0.8,0.8888888888888888\n90100051,6,1,2,0.8571428571428571,0.75,0.8\n90100052,4,1,2,0.8,0.6666666666666666,0.7272727272727273\n90100053,7,1,1,0.875,0.875,0.875\n90100054,8,0,0,1.0,1.0,1.0\n90100056,5,0,0,1.0,1.0,1.0\n90100057,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100058,7,0,0,1.0,1.0,1.0\n90100059,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100060,10,0,0,1.0,1.0,1.0\n90100061,6,0,0,1.0,1.0,1.0\n90100062,5,0,0,1.0,1.0,1.0\n90100064,6,0,0,1.0,1.0,1.0\n90100065,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100066,4,0,2,1.0,0.6666666666666666,0.8\n90100067,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100068,4,0,2,1.0,0.6666666666666666,0.8\n90100069,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n90100070,5,0,0,1.0,1.0,1.0\n90100072,3,0,1,1.0,0.75,0.8571428571428571\n90100073,7,0,0,1.0,1.0,1.0\n90100074,5,0,0,1.0,1.0,1.0\n90100075,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100076,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100077,5,0,0,1.0,1.0,1.0\n90100078,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n90100081,5,0,0,1.0,1.0,1.0\n90100082,5,0,0,1.0,1.0,1.0\n90100083,5,2,1,0.7142857142857143,0.8333333333333334,0.7692307692307693\n90100084,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n90100086,4,0,0,1.0,1.0,1.0\n12100000,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100001,7,1,0,0.875,1.0,0.9333333333333333\n12100002,8,0,0,1.0,1.0,1.0\n12100003,4,1,3,0.8,0.5714285714285714,0.6666666666666666\n12100004,9,0,0,1.0,1.0,1.0\n12100005,8,0,0,1.0,1.0,1.0\n12100006,7,0,0,1.0,1.0,1.0\n12100007,6,0,0,1.0,1.0,1.0\n12100008,7,0,0,1.0,1.0,1.0\n12100009,6,0,0,1.0,1.0,1.0\n12100010,5,0,0,1.0,1.0,1.0\n12100011,5,0,0,1.0,1.0,1.0\n12100012,5,0,0,1.0,1.0,1.0\n12100013,7,1,0,0.875,1.0,0.9333333333333333\n12100014,6,0,0,1.0,1.0,1.0\n12100015,6,0,0,1.0,1.0,1.0\n12100016,7,0,1,1.0,0.875,0.9333333333333333\n12100017,6,1,1,0.8571428571428571,0.8571428571428571,0.8571428571428571\n12100018,6,0,0,1.0,1.0,1.0\n12100019,8,0,0,1.0,1.0,1.0\n12100020,8,0,0,1.0,1.0,1.0\n12100021,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100022,7,0,0,1.0,1.0,1.0\n12100023,6,0,0,1.0,1.0,1.0\n12100024,5,0,0,1.0,1.0,1.0\n12100025,5,0,0,1.0,1.0,1.0\n12100026,7,0,0,1.0,1.0,1.0\n12100027,7,0,0,1.0,1.0,1.0\n12100028,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12100029,6,0,0,1.0,1.0,1.0\n12100030,6,0,0,1.0,1.0,1.0\n12100031,7,0,0,1.0,1.0,1.0\n12100032,8,0,0,1.0,1.0,1.0\n12100033,6,0,0,1.0,1.0,1.0\n12100034,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12100035,7,0,0,1.0,1.0,1.0\n12100036,5,0,0,1.0,1.0,1.0\n12100037,5,0,0,1.0,1.0,1.0\n12100038,6,0,0,1.0,1.0,1.0\n12100039,6,0,0,1.0,1.0,1.0\n12100040,4,0,0,1.0,1.0,1.0\n12100041,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12100042,6,0,0,1.0,1.0,1.0\n12100043,5,0,0,1.0,1.0,1.0\n12100044,7,0,0,1.0,1.0,1.0\n12100045,5,0,0,1.0,1.0,1.0\n12100046,6,0,0,1.0,1.0,1.0\n12100047,6,0,0,1.0,1.0,1.0\n12100048,6,0,0,1.0,1.0,1.0\n12100049,4,0,0,1.0,1.0,1.0\n12100050,4,1,0,0.8,1.0,0.8888888888888888\n12100051,6,0,0,1.0,1.0,1.0\n12100052,4,0,0,1.0,1.0,1.0\n12100053,5,2,1,0.7142857142857143,0.8333333333333334,0.7692307692307693\n12100054,5,0,0,1.0,1.0,1.0\n12100055,7,0,0,1.0,1.0,1.0\n12100056,5,0,0,1.0,1.0,1.0\n12100057,6,0,0,1.0,1.0,1.0\n12100058,5,0,0,1.0,1.0,1.0\n12100059,7,0,0,1.0,1.0,1.0\n12100060,8,0,0,1.0,1.0,1.0\n12100061,5,0,0,1.0,1.0,1.0\n12100062,6,0,0,1.0,1.0,1.0\n12100063,7,0,0,1.0,1.0,1.0\n12100064,5,0,0,1.0,1.0,1.0\n12100065,6,0,0,1.0,1.0,1.0\n12100121,7,0,0,1.0,1.0,1.0\n12100122,5,0,0,1.0,1.0,1.0\n12100123,4,0,0,1.0,1.0,1.0\n12100124,5,0,0,1.0,1.0,1.0\n12100125,6,0,0,1.0,1.0,1.0\n12100126,7,0,0,1.0,1.0,1.0\n12100127,3,0,0,1.0,1.0,1.0\n12100128,5,0,0,1.0,1.0,1.0\n12100129,7,0,0,1.0,1.0,1.0\n12100130,4,0,1,1.0,0.8,0.8888888888888888\n12100131,6,0,0,1.0,1.0,1.0\n12100132,6,0,0,1.0,1.0,1.0\n12100133,6,0,0,1.0,1.0,1.0\n12100134,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12100135,6,0,0,1.0,1.0,1.0\n12100136,7,1,0,0.875,1.0,0.9333333333333333\n12100137,7,0,0,1.0,1.0,1.0\n12100138,6,0,0,1.0,1.0,1.0\n12100139,5,0,0,1.0,1.0,1.0\n12100140,6,0,0,1.0,1.0,1.0\n12100141,7,0,0,1.0,1.0,1.0\n12100142,7,0,0,1.0,1.0,1.0\n12100143,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100144,7,0,0,1.0,1.0,1.0\n12100145,6,0,2,1.0,0.75,0.8571428571428571\n12100146,7,0,0,1.0,1.0,1.0\n12100147,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100148,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12100149,6,0,0,1.0,1.0,1.0\n12100150,6,0,0,1.0,1.0,1.0\n12100151,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100152,6,0,0,1.0,1.0,1.0\n12100153,6,0,0,1.0,1.0,1.0\n12100154,6,0,0,1.0,1.0,1.0\n12100155,5,0,0,1.0,1.0,1.0\n12100156,5,0,0,1.0,1.0,1.0\n12100157,7,0,0,1.0,1.0,1.0\n12100158,7,0,0,1.0,1.0,1.0\n12100159,8,0,1,1.0,0.8888888888888888,0.9411764705882353\n12100160,9,0,0,1.0,1.0,1.0\n12100161,7,0,0,1.0,1.0,1.0\n12100162,8,0,0,1.0,1.0,1.0\n12100163,7,0,0,1.0,1.0,1.0\n12100164,8,0,0,1.0,1.0,1.0\n12100165,8,0,0,1.0,1.0,1.0\n12100166,6,0,0,1.0,1.0,1.0\n12100167,8,0,0,1.0,1.0,1.0\n12100168,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100169,8,0,0,1.0,1.0,1.0\n12100170,6,0,0,1.0,1.0,1.0\n12100171,6,0,0,1.0,1.0,1.0\n12100172,6,0,0,1.0,1.0,1.0\n12100173,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100174,3,0,2,1.0,0.6,0.75\n12100175,6,0,0,1.0,1.0,1.0\n12100176,6,0,0,1.0,1.0,1.0\n12100177,5,0,0,1.0,1.0,1.0\n12100178,7,0,0,1.0,1.0,1.0\n12100179,7,0,0,1.0,1.0,1.0\n12100180,6,0,0,1.0,1.0,1.0\n12100181,5,0,0,1.0,1.0,1.0\n12100182,5,0,0,1.0,1.0,1.0\n12100183,6,0,0,1.0,1.0,1.0\n12100184,7,0,0,1.0,1.0,1.0\n12100185,5,0,0,1.0,1.0,1.0\n12100186,6,0,0,1.0,1.0,1.0\n12100187,7,0,0,1.0,1.0,1.0\n12100188,7,0,0,1.0,1.0,1.0\n12100189,7,0,1,1.0,0.875,0.9333333333333333\n12100190,6,0,0,1.0,1.0,1.0\n12100191,6,0,0,1.0,1.0,1.0\n12100192,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12100193,7,0,0,1.0,1.0,1.0\n12100194,6,0,0,1.0,1.0,1.0\n12100195,5,0,0,1.0,1.0,1.0\n12100196,5,0,0,1.0,1.0,1.0\n12100197,7,0,0,1.0,1.0,1.0\n12100198,7,0,0,1.0,1.0,1.0\n12100199,8,0,0,1.0,1.0,1.0\n12100200,10,0,0,1.0,1.0,1.0\n12300060,6,0,0,1.0,1.0,1.0\n12300061,7,1,0,0.875,1.0,0.9333333333333333\n12300062,5,0,0,1.0,1.0,1.0\n12300063,4,0,0,1.0,1.0,1.0\n12300064,6,0,0,1.0,1.0,1.0\n12300065,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12300066,5,0,0,1.0,1.0,1.0\n12300067,4,0,0,1.0,1.0,1.0\n12300068,5,0,0,1.0,1.0,1.0\n12300069,5,0,0,1.0,1.0,1.0\n12300070,7,0,0,1.0,1.0,1.0\n12300071,6,0,0,1.0,1.0,1.0\n12300072,8,0,0,1.0,1.0,1.0\n12300073,5,0,2,1.0,0.7142857142857143,0.8333333333333334\n12300074,7,0,0,1.0,1.0,1.0\n12300075,5,0,0,1.0,1.0,1.0\n12300076,3,0,2,1.0,0.6,0.75\n12300077,2,1,1,0.6666666666666666,0.6666666666666666,0.6666666666666666\n12300108,7,0,0,1.0,1.0,1.0\n12300109,9,0,0,1.0,1.0,1.0\n12300110,6,0,0,1.0,1.0,1.0\n12300111,6,0,0,1.0,1.0,1.0\n12300112,8,0,0,1.0,1.0,1.0\n12300113,6,0,0,1.0,1.0,1.0\n12300114,5,0,0,1.0,1.0,1.0\n12300115,6,0,0,1.0,1.0,1.0\n12300116,6,0,0,1.0,1.0,1.0\n12300117,5,0,0,1.0,1.0,1.0\n12300118,6,0,0,1.0,1.0,1.0\n12300119,6,0,0,1.0,1.0,1.0\n12300120,9,0,0,1.0,1.0,1.0\n12300121,8,0,0,1.0,1.0,1.0\n12300122,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n12300123,6,0,0,1.0,1.0,1.0\n12300124,7,0,0,1.0,1.0,1.0\n12300125,4,0,0,1.0,1.0,1.0\n12300126,6,0,0,1.0,1.0,1.0\n12300127,5,0,0,1.0,1.0,1.0\n12300128,6,0,0,1.0,1.0,1.0\n12300129,7,0,0,1.0,1.0,1.0\n12300130,6,0,0,1.0,1.0,1.0\n12300131,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300132,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300133,5,0,0,1.0,1.0,1.0\n12300134,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300135,6,0,0,1.0,1.0,1.0\n12300136,6,0,0,1.0,1.0,1.0\n12300137,5,0,0,1.0,1.0,1.0\n12300138,6,0,0,1.0,1.0,1.0\n12300139,6,0,0,1.0,1.0,1.0\n12300140,6,0,0,1.0,1.0,1.0\n12300141,8,1,0,0.8888888888888888,1.0,0.9411764705882353\n12300142,7,0,0,1.0,1.0,1.0\n12300144,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300145,6,2,0,0.75,1.0,0.8571428571428571\n12300146,6,0,0,1.0,1.0,1.0\n12300147,9,0,0,1.0,1.0,1.0\n12300148,5,0,0,1.0,1.0,1.0\n12300149,6,0,0,1.0,1.0,1.0\n12300150,9,0,0,1.0,1.0,1.0\n12300151,7,0,0,1.0,1.0,1.0\n12300152,7,0,0,1.0,1.0,1.0\n12300153,8,0,0,1.0,1.0,1.0\n12300154,6,0,1,1.0,0.8571428571428571,0.9230769230769231\n12300155,7,0,0,1.0,1.0,1.0\n12300156,7,0,0,1.0,1.0,1.0\n12300216,6,1,1,0.8571428571428571,0.8571428571428571,0.8571428571428571\n12300217,7,1,0,0.875,1.0,0.9333333333333333\n12300218,8,0,1,1.0,0.8888888888888888,0.9411764705882353\n12300219,5,0,0,1.0,1.0,1.0\n12300220,3,0,1,1.0,0.75,0.8571428571428571\n12300221,6,0,0,1.0,1.0,1.0\n12300222,3,0,2,1.0,0.6,0.75\n12300223,4,0,1,1.0,0.8,0.8888888888888888\n12300224,5,0,0,1.0,1.0,1.0\n12300225,7,0,0,1.0,1.0,1.0\n12300226,7,0,0,1.0,1.0,1.0\n12300227,4,1,1,0.8,0.8,0.8\n12300228,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12300229,5,0,0,1.0,1.0,1.0\n12300230,4,0,1,1.0,0.8,0.8888888888888888\n12300231,5,1,0,0.8333333333333334,1.0,0.9090909090909091\n12300232,5,0,0,1.0,1.0,1.0\n12300234,4,1,1,0.8,0.8,0.8\n12300235,5,0,0,1.0,1.0,1.0\n12300236,9,0,0,1.0,1.0,1.0\n12300237,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300238,7,0,0,1.0,1.0,1.0\n12300239,7,0,0,1.0,1.0,1.0\n12300240,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300241,6,2,0,0.75,1.0,0.8571428571428571\n12300242,7,0,0,1.0,1.0,1.0\n12300243,5,0,0,1.0,1.0,1.0\n12300244,5,0,0,1.0,1.0,1.0\n12300245,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12300246,7,0,0,1.0,1.0,1.0\n12300247,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n12300248,7,0,0,1.0,1.0,1.0\n12000001,7,5,2,0.5833333333333334,0.7777777777777778,0.6666666666666666\n12000017,7,5,0,0.5833333333333334,1.0,0.7368421052631579\n12000038,7,0,1,1.0,0.875,0.9333333333333333\n12000052,9,0,0,1.0,1.0,1.0\n12000068,6,3,0,0.6666666666666666,1.0,0.8\n12000085,4,10,1,0.2857142857142857,0.8,0.42105263157894735\n12000102,6,1,0,0.8571428571428571,1.0,0.9230769230769231\n12000119,6,2,0,0.75,1.0,0.8571428571428571\n12000136,5,0,1,1.0,0.8333333333333334,0.9090909090909091\n12000155,7,1,0,0.875,1.0,0.9333333333333333\n12000171,6,3,0,0.6666666666666666,1.0,0.8\n12000187,5,7,1,0.4166666666666667,0.8333333333333334,0.5555555555555556\n12000204,7,1,0,0.875,1.0,0.9333333333333333\n12000593,3,4,2,0.42857142857142855,0.6,0.5\n'))
history['gt_count']=history.TP+history.FN
history['pred_count']=history.TP+history.FP
print(pd.Series(summarize_group(history)))
print(pd.read_csv(io.StringIO('class,AP50,AP50_95\nopen,0.9769245917205036,0.6901412056054613\nshort,0.9254492801921426,0.6386143225534683\nmousebite,0.9786516922613111,0.74334660302292\nspur,0.9700825867239518,0.7268227217194034\ncopper,0.9906896929315278,0.8952728783876003\npin-hole,0.9906149091440183,0.8352701526976366\n')).to_string(index=False))
from pathlib import Path
historical_out=Path('historical_pcb_reports')
historical_out.mkdir(exist_ok=True)
pd.DataFrame([summarize_group(history)]).to_csv(historical_out/'yolo_inspection.csv', index=False)
(historical_out/'comparison_iou50.csv').write_text(',TP,FP,FN,precision,recall,f1,fp_per_image\nOpenCV baseline,2139,43874,1001,0.046486862408449785,0.6812101910828026,0.08703436209387017,87.748\nYOLO11n,2955,154,185,0.9504663879060792,0.9410828025477707,0.9457513202112338,0.308\n')
(historical_out/'per_class_ap.csv').write_text('class,AP50,AP50_95\nopen,0.9769245917205036,0.6901412056054613\nshort,0.9254492801921426,0.6386143225534683\nmousebite,0.9786516922613111,0.74334660302292\nspur,0.9700825867239518,0.7268227217194034\ncopper,0.9906896929315278,0.8952728783876003\npin-hole,0.9906149091440183,0.8352701526976366\n')
(historical_out/'validation_thresholds.csv').write_text('conf,TP,FP,FN,precision,recall,f1,fp_per_image\n0.05,1359,314,12,0.8123132098027496,0.9912472647702407,0.892904073587385,1.57\n0.1,1355,203,16,0.8697047496790757,0.9883296863603209,0.9252304540798908,1.015\n0.2,1348,131,23,0.9114266396213658,0.9832239241429613,0.9459649122807018,0.655\n0.3,1340,97,31,0.9324982602644398,0.9773887673231219,0.9544159544159544,0.485\n0.4,1331,73,40,0.948005698005698,0.9708242159008024,0.9592792792792793,0.365\n0.5,1317,56,54,0.9592134013109979,0.9606126914660832,0.9599125364431487,0.28\n0.65,1277,30,94,0.9770466717674062,0.9314369073668854,0.9536967886482449,0.15\n0.8,1149,10,222,0.99137187230371,0.838074398249453,0.908300395256917,0.05\n')
print('上次實測比較：')
print(pd.read_csv(historical_out/'comparison_iou50.csv',index_col=0).to_string())
print('歷史報告',historical_out.resolve())

## 1｜設定與權重
在 Colab 左側檔案面板上傳原成果 ZIP 至 /content。只載入你自己的權重。可選的 META 與 EXTERNAL 檔案格式見第 5 節。預設將重跑 500 張原圖與四種擾動；若只想核對原圖，設定 RUN_STRESS=False。

In [ ]:
%pip -q install ultralytics==8.4.26

In [ ]:
from pathlib import Path
from datetime import datetime
import json, shutil, platform
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch, ultralytics, yaml
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment

EPOCHS = 50
BATCH = 8
SIZE = 640
SEED = 20261117
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
ROOT = (Path.cwd() / ('pcb_run_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))).resolve()
DS, OUT = ROOT/'dataset', ROOT/'results'
OUT.mkdir(parents=True)
print(ROOT, DEVICE, 'torch', torch.__version__, 'ultralytics', ultralytics.__version__)
TRAIN_NEW=False
ARCHIVE=Path('/content/pcb_model_results.zip')
META=''             # 選用，例如 /content/deeppcb_conditions.csv
EXTERNAL=''         # 選用，例如 /content/factory/manifest.csv
RUN_STRESS=True
import zipfile, cv2, itertools, hashlib

## 2｜資料切分與雙模型
鎖定與前版相同的 commit，官方 trainval 分 800/200 張，官方 test 500 張保持獨立。OpenCV 以兩張對齊影像作灰階高斯平滑、絕對差分、二值化、閉運算和連通區分析，剔除小區域並向外擴 8 像素。YOLO 僅使用待測影像。兩者輸入需求不同，定位評分規則相同。

In [ ]:
import subprocess, hashlib
from scipy.optimize import linear_sum_assignment
REPO_URL = 'https://github.com/tangsanli5201/DeepPCB.git'
DATA_COMMIT = '08e98c4db5922613fb97176eb3d6497d48260cb1'
REPO = Path('DeepPCB').resolve()
if not (REPO / '.git').exists():
    REPO.mkdir(exist_ok=True)
    subprocess.run(['git', 'init', str(REPO)], check=True, capture_output=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'add', 'origin', REPO_URL], check=True)
current = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True)
if current.returncode or current.stdout.strip() != DATA_COMMIT:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', DATA_COMMIT], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', DATA_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == DATA_COMMIT
DATA = REPO / 'PCBData'

CLASSES = {1:'open',2:'short',3:'mousebite',4:'spur',5:'copper',6:'pin-hole'}
def read_split(filename):
    records=[]
    for line in (DATA/filename).read_text().splitlines():
        if not line.strip(): continue
        image, annotation = line.split()
        base = DATA/image
        ref = base.with_name(base.stem+'_temp.jpg')
        test_path = base.with_name(base.stem+'_test.jpg')
        ann = DATA/annotation
        for file in [ref,test_path,ann]:
            if not file.is_file(): raise FileNotFoundError(file)
        records.append(dict(id=base.stem, reference=ref, test=test_path, annotation=ann))
    return records

train_records = read_split('trainval.txt')
test_records = read_split('test.txt')
assert len(train_records)==1000 and len(test_records)==500
assert len({r['id'] for r in train_records})==1000
assert len({r['id'] for r in test_records})==500
assert not ({r['id'] for r in train_records} & {r['id'] for r in test_records})
rng = np.random.default_rng(20261117)
val_records = [train_records[i] for i in rng.permutation(len(train_records))[:200]]


val_ids = {r['id'] for r in val_records}
splits = {'train':[r for r in train_records if r['id'] not in val_ids],
          'val':val_records,'test':test_records}

In [ ]:
def target_rows(gt, w, h):
    x1,y1,x2,y2,label = gt.T
    if not (np.all(x1>=0) and np.all(y1>=0) and np.all(x2<w) and np.all(y2<h)
            and np.all(x2>=x1) and np.all(y2>=y1) and np.isin(label,list(CLASSES)).all()):
        raise ValueError('Invalid DeepPCB annotation')
    return np.column_stack([label-1,(x1+x2+1)/(2*w),(y1+y2+1)/(2*h),
                            (x2-x1+1)/w,(y2-y1+1)/h])

def load_gt(r):
    a = np.loadtxt(r['annotation'],dtype=int,ndmin=2)
    if a.shape[1]!=5: raise ValueError(str(r['annotation']))
    return a

def inclusive_boxes(xyxy):
    boxes = np.asarray(xyxy,dtype=float).reshape(-1,4).copy()
    boxes[:,2:] -= 1
    return boxes

In [ ]:
for split,records in splits.items():
    imgs,labels = DS/'images'/split, DS/'labels'/split
    imgs.mkdir(parents=True); labels.mkdir(parents=True)
    for r in records:
        with Image.open(r['test']) as im:
            rows = target_rows(load_gt(r), *im.size)
        name = r['id']
        shutil.copyfile(r['test'],imgs/(name+'.jpg'))
        np.savetxt(labels/(name+'.txt'),rows,fmt=['%d']+['%.9f']*4)
    print(split,len(records))
cfg = dict(path=str(DS),train='images/train',val='images/val',test='images/test',
           names={i-1:c for i,c in CLASSES.items()})
YAML = DS/'pcb.yaml'
YAML.write_text(yaml.safe_dump(cfg,sort_keys=False))
manifest = dict(commit=DATA_COMMIT,seed=SEED,ids={s:[r['id'] for r in rs] for s,rs in splits.items()},
                torch=str(torch.__version__),ultralytics=ultralytics.__version__,python=platform.python_version())
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))

In [ ]:
if TRAIN_NEW:
    trainer=YOLO('yolo11n.pt')
    trainer.train(data=str(YAML),epochs=EPOCHS,imgsz=SIZE,batch=BATCH,device=DEVICE,seed=SEED,
                  workers=2,patience=12,project=str(ROOT),name='training',plots=True)
    best=Path(trainer.trainer.save_dir)/'weights'/'best.pt'
else:
    if not ARCHIVE.is_file(): raise FileNotFoundError('請上傳原成果 ZIP，或改 TRAIN_NEW=True')
    with zipfile.ZipFile(ARCHIVE) as payload:
        saved=json.loads(payload.read('manifest.json'))
        if saved['commit']!=DATA_COMMIT or saved['ids']!=manifest['ids']:
            raise ValueError('原權重的資料清單與本次切分不同')
        best=ROOT/'best.pt'
        best.write_bytes(payload.read('best.pt'))
model=YOLO(str(best))
if model.names!={i-1:c for i,c in CLASSES.items()}: raise ValueError('權重類別順序不符')
print('Weights',hashlib.sha256(best.read_bytes()).hexdigest())

In [ ]:
def read_bgr(path):
    im=cv2.imread(str(path),cv2.IMREAD_COLOR)
    if im is None: raise ValueError(f'無法讀取 {path}')
    return im

def load_boxes(path,w,h):
    if path is None or not path.read_text().strip():
        return np.empty((0,5),dtype=int)
    g=np.loadtxt(path,ndmin=2)
    if g.shape[1]!=5 or not np.isfinite(g).all() or not np.array_equal(g,g.astype(int)):
        raise ValueError(f'標註格式錯誤 {path}')
    target_rows(g,w,h)
    return g.astype(int)

def read_conditions(path):
    if not path: return {}
    df=pd.read_csv(path,dtype=str,keep_default_na=False)
    if not {'id','material','lighting'}<=set(df) or df.id.duplicated().any():
        raise ValueError('metadata 欄位需含唯一 id、material、lighting')
    if set(df.id)-set(r['id'] for r in test_records): raise ValueError('metadata 有未知 DeepPCB test ID')
    return df.set_index('id')[['material','lighting']].to_dict('index')

def external_records(path):
    if not path: return []
    root=Path(path).resolve().parent
    df=pd.read_csv(path,dtype=str,keep_default_na=False)
    fields={'id','test','reference','annotation','truth','material','lighting'}
    if not fields<=set(df) or df.id.duplicated().any() or df.empty:
        raise ValueError('外部 CSV 缺欄、重複 ID 或空白')
    if set(df.id)&{r['id'] for rows in splits.values() for r in rows}:
        raise ValueError('外部 ID 與 DeepPCB 重複，請使用獨立檢驗品')
    records=[]
    for d in df.to_dict('records'):
        for k in ('test','reference','annotation'):
            d[k]=(root/d[k]).resolve() if d[k] else None
        if d['truth'] not in ('OK','NG') or not d['test'] or not d['reference']:
            raise ValueError(f"缺少影像或真值 {d['id']}")
        if d['test']==d['reference']: raise ValueError('模板與待測品必須是獨立影像')
        if d['truth']=='NG' and not d['annotation']: raise ValueError('NG 需完整框標註')
        records.append(d)
    return records

def perturb(img,condition):
    if condition=='original': return img
    a=img.astype(np.float32)
    if condition=='dark_065': a*=0.65
    elif condition=='bright_125_plus15': a=a*1.25+15
    elif condition=='gradient_055_135':
        a*=np.linspace(.55,1.35,img.shape[1],dtype=np.float32)[None,:,None]
    elif condition=='sim_surface_glare':
        h,w=a.shape[:2]
        yy,xx=np.ogrid[:h,:w]
        veil=.65*np.exp(-((xx-.58*w)/(.18*w))**2-((yy-.43*h)/(.23*h))**2)
        a=a*(1-veil[:,:,None])+255*veil[:,:,None]
    else: raise ValueError(condition)
    return np.uint8(np.clip(a,0,255))

def classic(ref,img,pixel,area):
    gray=lambda a:cv2.GaussianBlur(cv2.cvtColor(a,cv2.COLOR_BGR2GRAY),(3,3),0)
    diff=cv2.absdiff(gray(ref),gray(img))
    raw=cv2.threshold(diff,pixel,255,cv2.THRESH_BINARY)[1]
    mask=cv2.morphologyEx(raw,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))
    stats=cv2.connectedComponentsWithStats(mask,connectivity=8)[2][1:]
    h,w=img.shape[:2]
    return np.array([[max(0,x-8),max(0,y-8),min(w-1,x+bw-1+8),min(h-1,y+bh-1+8)]
                     for x,y,bw,bh,size in stats if size>=area],dtype=float).reshape(-1,4)

def records_in_batches(records,condition):
    for start in range(0,len(records),BATCH):
        batch=[]
        for r in records[start:start+BATCH]:
            ref,im=read_bgr(r['reference']),read_bgr(r['test'])
            if ref.shape!=im.shape: raise ValueError(f"模板尺寸不符 {r['id']}")
            h,w=im.shape[:2]
            gt=load_boxes(r['annotation'],w,h)
            if 'truth' in r and (r['truth']=='NG')!=bool(len(gt)): raise ValueError(f"真值與框不符 {r['id']}")
            batch.append((r,ref,perturb(im,condition),gt))
        yield batch

def yolo_batch(batch):
    results=model.predict([e[2] for e in batch],conf=.01,iou=.7,imgsz=SIZE,device=DEVICE,verbose=False)
    for res in results:
        b=res.boxes
        yield inclusive_boxes(b.xyxy.cpu().numpy()),b.conf.cpu().numpy(),b.cls.int().cpu().numpy()+1

## 3｜僅以驗證集選門檻
在未擾動的 200 張驗證圖比較 OpenCV 的差分門檻 × 面積門檻與 YOLO 信心門檻。兩者均以整體框定位 F1 最大化。分數相同時 YOLO 取較高信心，OpenCV 保留固定候選順序。框匹配採含端點座標、嚴格 IoU > 0.50，一對一且先最大化匹配數；NMS IoU 另固定為 0.7。所有光照及外部材質用同一組鎖定參數。

In [ ]:
cv_params=list(itertools.product([20,50],[10,25]))
conf_grid=[.05,.10,.20,.30,.40,.50,.65,.80]
counts={k:np.zeros(3,dtype=int) for k in cv_params+conf_grid}
for batch in records_in_batches(val_records,'original'):
    for (_,ref,img,gt),(box,conf,_) in zip(batch,yolo_batch(batch),strict=True):
        candidates=[(k,classic(ref,img,*k)) for k in cv_params]+[(k,box[conf>=k]) for k in conf_grid]
        for k,p in candidates:
            pairs,fp,fn=match_boxes(p,gt,.5)
            counts[k]+=[len(pairs),fp,fn]
val={k:scores(*map(int,v)) for k,v in counts.items()}
selected_cv=max(cv_params,key=lambda k:val[k]['f1'])
selected_conf=max(conf_grid,key=lambda k:(val[k]['f1'],k))
pd.DataFrame([dict(candidate=str(k),method='OpenCV' if isinstance(k,tuple) else 'YOLO11n',**v)
              for k,v in val.items()]).to_csv(OUT/'validation_grid.csv',index=False)
lock=dict(cv_threshold=selected_cv[0],cv_min_area=selected_cv[1],yolo_conf=selected_conf,
          nms_iou=.7,match='one-to-one IoU > 0.50 inclusive xyxy',validation_ids=sorted(val_ids),
          split_seed=SEED,weight_sha256=hashlib.sha256(best.read_bytes()).hexdigest())
(OUT/'locked_thresholds.json').write_text(json.dumps(lock,indent=2))
print(selected_cv,selected_conf)

## 4｜正式測試與逐類 AP
mAP 衡量類別與位置，在多個信心門檻上積分；後續生產品質指標使用鎖定的單一信心門檻。傳統差分沒有類別輸出，逐類僅統計各真實瑕疵被定位的比例。

In [ ]:
ap=model.val(data=str(YAML),split='test',imgsz=SIZE,batch=BATCH,device=DEVICE,
             conf=.001,iou=.7,project=str(ROOT),name='test_map',plots=True)
pd.DataFrame([dict(defect=model.names[int(c)],AP50=float(ap.box.ap50[i]),AP50_95=float(ap.box.ap[i]))
              for i,c in enumerate(ap.box.ap_class_index)]).to_csv(OUT/'per_class_ap.csv',index=False)
(OUT/'map.json').write_text(json.dumps(dict(mAP50=float(ap.box.map50),mAP50_95=float(ap.box.map))))

## 5｜光線、表面反光與真實材質
每張官方測試圖分別評估原圖、亮度 ×0.65、亮度 ×1.25+15、水平亮度漸層與局部亮斑。擾動只套在待測圖，模擬建模板之後照明發生變化。幾何位置與標註不動，同一影像配對比較。亮斑為光學外觀的合成壓力測試，無法據此辨識基材。勿把不同擾動視作更多獨立檢驗品。

**真實材質／實拍光源**　若已有官方測試圖的可靠製程紀錄，提供 META CSV，欄位為 id,material,lighting；缺資料顯示 unknown。若增加獨立實拍資料，提供 EXTERNAL CSV，建議每種材質 × 光源皆含良品與瑕疵品，各實體工件只使用一次。欄位和示意值如下，檔案路徑相對 CSV；材質僅填可查證的名稱。外部 NG 框需使用同一套六類定義。

```csv
id,test,reference,annotation,truth,material,lighting
A001,images/A001.jpg,templates/T01.jpg,,OK,FR4,ring
A002,images/A002.jpg,templates/T01.jpg,labels/A002.txt,NG,FR4,ring
B001,images/B001.jpg,templates/T02.jpg,,OK,alumina,diffuse
```

每個 NG 標註檔一行一框，以空白分隔 x1 y1 x2 y2 class_id（含端點，類別 1–6）。OK 由人工檢驗真值提供，參考模板是另一張已對齊的影像。外部集只測試，不參與門檻選擇。材質間若光源、板型或瑕疵組成不同，分層差異僅供描述，不能單獨歸因於材質。

In [ ]:
def analyze(records,source,condition):
    for batch in records_in_batches(records,condition):
        for (r,ref,img,gt),(boxes,conf,labels) in zip(batch,yolo_batch(batch),strict=True):
            keep=conf>=selected_conf
            for method,p,cls in [('OpenCV',classic(ref,img,*selected_cv),None),
                                  ('YOLO11n',boxes[keep],labels[keep])]:
                pairs,fp,fn=match_boxes(p,gt,.5)
                key=dict(source=source,condition=condition,method=method,id=r['id'],
                         material=r.get('material') or 'unknown',lighting=r.get('lighting') or 'unknown')
                image_rows.append(dict(**key,TP=len(pairs),FP=fp,FN=fn,gt_count=len(gt),pred_count=len(p)))
                matched_gt={j for _,j in pairs}
                for c,name in CLASSES.items():
                    g=gt[gt[:,4]==c]
                    entry=dict(**key,defect=name,gt=len(g),localized=sum(gt[j,4]==c for j in matched_gt))
                    if cls is not None:
                        hits,extra,miss=match_boxes(p[cls==c],g,.5)
                        entry.update(class_tp=len(hits),class_fp=extra,class_fn=miss)
                    class_rows.append(entry)
                if method=='YOLO11n' and (fp or fn) and source=='DeepPCB' and condition=='original' and len(examples)<4:
                    left,right=img.copy(),img.copy()
                    for panel,rects in ((left,gt[:,:4]),(right,p)):
                        for x1,y1,x2,y2 in rects.astype(int):
                            cv2.rectangle(panel,(x1,y1),(x2,y2),(0,190,255),2)
                    dest=OUT/f"error_{r['id']}_GT_left_pred_right.png"
                    cv2.imwrite(str(dest),np.hstack([left,right]))
                    examples.append(r['id'])

In [ ]:
conditions=read_conditions(META)
for r in test_records: r.update(conditions.get(r['id'],{}))
others=external_records(EXTERNAL)
image_rows,class_rows,examples=[],[],[]
variants=['original']+(['dark_065','bright_125_plus15','gradient_055_135','sim_surface_glare'] if RUN_STRESS else [])
for variant in variants:
    analyze(test_records,'DeepPCB',variant)
    print('完成',variant)
if others: analyze(others,'external','original')

## 6｜良率、漏檢與誤報報告
真值良率 = 真良品 / 全部檢驗品；預測放行率 = 判 OK / 全部；整件漏放率 = NG 被判 OK / 真 NG；良品誤剔率 = OK 被判 NG / 真 OK。框漏檢率 = FN / (TP+FN)；每圖誤報 = FP / 張數。分母為零留空。模型在錯位置報警也可能讓 NG 件被攔下，因此整件漏放與框漏檢須並列。真值良率反映測試樣本組成；估計產線良率需連續批次或具代表性的抽樣。

In [ ]:
images=pd.DataFrame(image_rows)
byclass=pd.DataFrame(class_rows)
base=['source','condition','method']
images.to_csv(OUT/'per_image.csv',index=False)
summary=grouped(images,base,summarize_group)
summary.to_csv(OUT/'inspection_report.csv',index=False)
material=grouped(images.query("condition=='original'"),['source','method','material','lighting'],summarize_group)
material.to_csv(OUT/'material_lighting_report.csv',index=False)
def class_summary(g):
    n,h=int(g.gt.sum()),int(g.localized.sum())
    row=dict(gt=n,localized=h,missed=n-h,localization_recall=ratio(h,n))
    if g.method.iloc[0]=='YOLO11n':
        tp,fp,fn=[int(g[k].sum()) for k in ['class_tp','class_fp','class_fn']]
        row.update(class_tp=tp,class_fp=fp,class_fn=fn,class_precision=ratio(tp,tp+fp),class_recall=ratio(tp,tp+fn))
    return row
grouped(byclass,base+['defect'],class_summary).to_csv(OUT/'defect_report.csv',index=False)
original=summary.query("condition=='original'")[['source','method','recall','fp_per_image']]
delta=summary.merge(original,on=['source','method'],suffixes=('','_original'),validate='many_to_one')
delta['recall_change_pp']=100*(delta.recall-delta.recall_original)
delta['extra_fp_per_image']=delta.fp_per_image-delta.fp_per_image_original
delta.to_csv(OUT/'stress_comparison.csv',index=False)
print(summary[['source','method','condition','gt_yield','pass_rate','escape_rate','false_reject_rate','FN','FP','recall','fp_per_image']].to_string(index=False))
print(material[['material','lighting','method','images','gt_good','gt_bad','escape_rate','false_reject_rate']].to_string(index=False))

In [ ]:
import shutil
if TRAIN_NEW: shutil.copy2(best,OUT/'best.pt')
archive=shutil.make_archive(str(ROOT/'pcb_full_report'),'zip',OUT)
print('完整報告',archive)
# from google.colab import files
# files.download(archive)

**讀報告的順序**　開 inspection_report.csv 先看 original 的真 NG 與真 OK 分母，再比兩法的 FN、FP 與整件誤判。看 defect_report.csv 找出最容易漏掉的瑕疵；用 stress_comparison.csv 查召回率變化（百分點）和每圖多出的誤報框。最後看 material_lighting_report.csv 是否有足夠真實樣本覆蓋預計上線的條件。DeepPCB 已被用於前版成果，正式導入前宜另保留未參與歷次開發的新批次。